# 02 — Lọc và chuẩn bị
Chọn ngẫu nhiên `N_MOLECULES` phân tử (≤5 nguyên tử nặng, seed 42) từ các ứng viên đã qua kiểm tra, chia train/val/test **theo phân tử**, và thêm 5 nguyên tử cô lập (H, C, N, O, F). Nguyên tử luôn nằm trong tập train: chúng cần để tính năng lượng nguyên tử hóa, thước đo dùng ở bước benchmark.

Loại: phân tử trong `uncharacterized.txt`, phân tử có số electron lẻ (DFT/CCSD(T) vỏ kín không áp dụng).

**Đầu ra:** `data/processed/systems.csv` (mọi hệ cần tính), `molecules_xyz/`, `selection_report.json`.

In [ ]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ml_xc_colab")
else:                                    # chạy cục bộ (VS Code/Jupyter): notebook nằm trong notebooks/
    here = Path(os.environ.get("ML_XC_ROOT", Path.cwd())).resolve()
    ROOT = next((p for p in (here, here.parent) if (p / "utils").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("Không thấy thư mục utils/. Mở notebook trong ml_xc_colab/notebooks hoặc đặt ML_XC_ROOT.")
sys.path.insert(0, str(ROOT))
ROOT = str(ROOT)
DATA, RAW, PROC, CKPT = (f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed", f"{ROOT}/data/checkpoints")
for d in (RAW, PROC, CKPT):
    os.makedirs(d, exist_ok=True)
try:                                     # giới hạn bộ nhớ PySCF = 70% RAM (MB); đặt trước khi import pyscf
    import psutil
    os.environ.setdefault("PYSCF_MAX_MEMORY", str(int(psutil.virtual_memory().total / 1e6 * 0.7)))
except ImportError:
    pass
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd
from utils import data_utils as du
from utils import dft_utils as dft

N_MOLECULES = 50
SEED = 42
FRACTIONS = (0.6, 0.2, 0.2)       # train / val / test, chia theo phân tử

In [ ]:
cands = [json.loads(l) for l in open(f"{RAW}/qm9_candidates.jsonl")]
bad = du.read_uncharacterized(f"{RAW}/uncharacterized.txt")
report = {"n_candidates": len(cands)}

ok = [c for c in cands if c["gdb_id"] not in bad]
report["after_uncharacterized"] = len(ok)
ok = [c for c in ok if du.n_electrons(c["symbols"]) % 2 == 0 and set(c["symbols"]) <= set(du.ELEMENTS)]
report["after_closed_shell_CHNOF"] = len(ok)
print(report)

rng = np.random.default_rng(SEED)
pick = [ok[i] for i in sorted(rng.choice(len(ok), size=min(N_MOLECULES, len(ok)), replace=False))]
names = [f"gdb_{c['gdb_id']}" for c in pick]
split = du.random_split(names, FRACTIONS, seed=SEED)

In [ ]:
os.makedirs(f"{PROC}/molecules_xyz", exist_ok=True)
rows = []
for c, name in zip(pick, names):
    du.write_xyz(f"{PROC}/molecules_xyz/{name}.xyz", c["symbols"], c["coords_A"], f"{name} {c['smiles']}")
    rows.append(dict(name=name, split=split[name], is_atom=False, symbol="", n_atoms=c["n_atoms"], n_heavy=c["n_heavy"],
                     formula=c["formula"], smiles=c["smiles"], gdb_id=c["gdb_id"], U0_B3LYP_Ha=c["U0_B3LYP_Ha"],
                     nelec=du.n_electrons(c["symbols"]), spin=0,
                     symbols_json=json.dumps(c["symbols"]), coords_json=json.dumps(c["coords_A"]),
                     counts_json=json.dumps(du.composition(c["symbols"]))))
for el in du.ELEMENTS:                  # nguyên tử cô lập, luôn thuộc train, spin cơ bản
    rows.append(dict(name=f"atom_{el}", split="train", is_atom=True, symbol=el, n_atoms=1, n_heavy=int(el != "H"),
                     formula=el, smiles="", gdb_id=-1, U0_B3LYP_Ha=np.nan, nelec=du.ATOMIC_NUMBER[el],
                     spin=dft.ATOM_SPIN[el], symbols_json=json.dumps([el]), coords_json=json.dumps([[0.0, 0.0, 0.0]]),
                     counts_json=json.dumps({el: 1})))
systems = pd.DataFrame(rows)
systems.to_csv(f"{PROC}/systems.csv", index=False)

mol = systems[~systems.is_atom]
report.update(n_molecules=int(len(mol)), split_counts=mol.split.value_counts().to_dict(), seed=SEED,
              n_heavy_counts=mol.n_heavy.value_counts().sort_index().to_dict(),
              element_counts={e: int(sum(json.loads(c).get(e, 0) > 0 for c in mol.counts_json)) for e in du.ELEMENTS})
json.dump(report, open(f"{PROC}/selection_report.json", "w"), indent=2)
print(json.dumps(report, indent=1))
missing = [e for e, k in report["element_counts"].items() if k == 0]
if missing:
    print("LƯU Ý: không có phân tử nào chứa", missing, "(chỉ có nguyên tử cô lập của chúng trong tập)")
mol[["name", "split", "formula", "n_heavy", "n_atoms", "smiles"]].head(15)